# 01 · Load and Clean

**Goal:** load the raw Olist CSVs into a SQLite database, check data quality,
and build one clean, order-level analysis table for the rest of the project.

Data: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
(CC BY-NC-SA 4.0). Download the CSVs into `data/` before running this notebook.

## 1. Load raw data                                      

### 1.1 Load CSVs into SQLite (rebuild olist.db)

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import empiricaldist

In [2]:
DATA_DIR = Path("../data")
DB_PATH = DATA_DIR / "olist.db"

DB_PATH.unlink(missing_ok=True)   # rebuild from scratch
conn = sqlite3.connect(DB_PATH)

csv_paths = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_paths)} CSV files")

Found 9 CSV files


In [3]:
def table_name_from(path: Path) -> str:
    """olist_orders_dataset.csv -> orders"""
    return path.stem.replace("olist_", "").replace("_dataset", "")

for path in csv_paths:
    df = pd.read_csv(path)
    new_name = table_name_from(path)
    globals()[new_name] = df
    df.to_sql(table_name_from(path), conn, if_exists="replace", index=False)

print("Loaded tables:", [table_name_from(p) for p in csv_paths])

Loaded tables: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'product_category', 'products', 'sellers']


### 1.2 Verify the load (SQL rows vs. file lines)
Compare each table's SQL row count with the raw line count of its CSV file.
A mismatch means the file has line breaks *inside* quoted fields.

In [4]:
print(f"{'TABLE':<35} {'SQL ROWS':>10} {'FILE LINES':>11}")
print("")
for path in csv_paths:
    table = table_name_from(path)
    sql_rows = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"].iloc[0]
    with open(path, encoding="utf-8") as f:
        file_lines = sum(1 for _ in f) - 1      # minus header
    flag = "  <-- mismatch" if sql_rows != file_lines else ""
    print(f"{table:<35} {sql_rows:>10,} {file_lines:>11,}{flag}")

TABLE                                 SQL ROWS  FILE LINES

customers                               99,441      99,441
geolocation                          1,000,163   1,000,163
order_items                            112,650     112,650
order_payments                         103,886     103,886
order_reviews                           99,224     104,719  <-- mismatch
orders                                  99,441      99,441
product_category                            71          71
products                                32,951      32,951
sellers                                  3,095       3,095


In [5]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Table in olist.db:")
for table in tables:
    query = f"SELECT count(*) FROM {table[0]};"
    row = cursor.execute(query).fetchone()
    print(f"{table[0]:<35} has {row[0]} rows")

Table in olist.db:
customers                           has 99441 rows
geolocation                         has 1000163 rows
order_items                         has 112650 rows
order_payments                      has 103886 rows
order_reviews                       has 99224 rows
orders                              has 99441 rows
product_category                    has 71 rows
products                            has 32951 rows
sellers                             has 3095 rows


All tables match their files except `order_reviews` (99,224 rows vs. 104,719 lines):
review comments contain line breaks, so one row can span several lines.
The SQL row count is the correct one.

## 2. Data quality checks

### 2.1 Reviews: Overview

`order_reviews` has 99,224 rows but `orders` has 99,441. Before joining,
check whether any order has more than one review (which would duplicate
orders in the join).

In [6]:
query = """
SELECT
    (SELECT COUNT(*) FROM orders) AS total_orders,
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT order_id) AS total_reviewed_orders,
    COUNT(*) - COUNT(DISTINCT order_id) AS gap_review_orders,
    (SELECT COUNT(*) FROM (
        SELECT order_id
        FROM order_reviews
        GROUP BY order_id
        HAVING COUNT(*) > 1
    ) AS multiple_review_orders) AS orders_with_mreviews,
    (SELECT COUNT(*) FROM orders) - COUNT(DISTINCT order_id) AS orders_without_reviews
    
FROM order_reviews
"""
reviews_gap = pd.read_sql_query(query, conn)
reviews_gap

,total_orders,total_reviews,total_reviewed_orders,gap_review_orders,orders_with_mreviews,orders_without_reviews
0,99441,99224,98673,551,547,768


List of Orders with multiple reviews

In [7]:
query = """
SELECT 
    order_id, 
    COUNT(*) AS total_reviews
FROM order_reviews
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY total_reviews DESC;
"""
result = pd.read_sql_query(query, conn)
result.head(5)

,order_id,total_reviews
0,df56136b8031ecd28e200bb18e6ddb2e,3
1,c88b1d1b157a9999ce368f218a407141,3
2,8e17072ec97ce29f0e1f111e598b0c85,3
3,03c939fd7fd3b38f8485a0f95798f1f6,3
4,ffaabba06c9d293a3c614e0515ddbabc,2


1. There 768 orders have no review. (0.77%)
2. There are 547 orders have more than 1 review (0.55%) . There are 4 orders have 3 reviews and 543 orders have 2 reviews.

### 2.2 Reviews: Duplicate reviews per order

In [8]:
query = """
WITH multi AS (
    SELECT order_id
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
),
with_prev AS (
    SELECT
        order_id,
        review_score,
        LAG(review_score) OVER (
            PARTITION BY order_id
            ORDER BY review_answer_timestamp
        ) AS prev_score
    FROM order_reviews
    WHERE order_id IN (SELECT order_id FROM multi)
)
SELECT
    COUNT(*)                            AS review_pairs,
    SUM(review_score <  prev_score)     AS lower,
    SUM(review_score =  prev_score)     AS same,
    SUM(review_score >  prev_score)     AS higher,
    ROUND(AVG(review_score - prev_score), 2) AS avg_change
FROM with_prev
WHERE prev_score IS NOT NULL;

"""
result = pd.read_sql_query(query, conn)
result


,review_pairs,lower,same,higher,avg_change
0,551,114,348,89,-0.15


Same score review is the most portion in 551 review pairs

 **Finding:** 
1. There 768 orders have no review. (0.77%)
2. There are 547 orders have more than 1 review (0.55%) . There are 4 orders have 3 reviews and the rest has 2 reviews.

|  | total_orders | total_reviews | total_reviewed_orders | gap_review_orders | orders_with_mreviews | orders_without_reviews |
| --- | --- | --- | --- | --- | --- | --- |
| **0** | 99441 | 99224 | 98673 | 551 | 547 | 768 |



3. Out of 551 consecutive review pairs, 348 (63%) kept the same score,
114 (21%) dropped and 89 (16%) rose (average change −0.15 points).
Most duplicate reviews agree; when they differ, drops are slightly more common.

|  | review_pairs | lower | same | higher | avg_change |
| --- | --- | --- | --- | --- | --- |
| **0** | 551 | 114 | 348 | 89 | -0.15 |


**Decision:** 

Keep only the **latest** review per order (by `review_answer_timestamp`)
as the customer's final opinion. This affects fewer than 0.3% of orders, so the
choice does not materially change later results. Orders without a review are kept
for delivery analysis but excluded from review-score tests (H1).

In [9]:
query = """
With rank as (
    select *, row_number() over (partition by order_id order by review_answer_timestamp desc) as row_num
    from order_reviews)
select * from rank where row_num =1

"""
order_reviews_new = pd.read_sql_query(query, conn)
order_reviews_new.to_sql('order_reviews_new', conn, if_exists='replace', index=False)
order_reviews_new.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,row_num
0,97ca439bc427b48bc1cd7177abe71365,00010242fe8c5a6d1ba2dd792cb16214,5,NaN,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,2017-09-22 10:57:03,1
1,7b07bacd811c4117b742569b04ce3580,00018f77f2f0320c557190d7a144bdd3,4,NaN,NaN,2017-05-13 00:00:00,2017-05-15 11:34:13,1
2,0c5b33dea94867d1ac402749e5438e8b,000229ec398224ef6ca0657da4fc703e,5,NaN,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,2018-01-23 16:06:31,1
3,f4028d019cb58564807486a6aaf33817,00024acbcdf0a6daa1e931b038114c75,4,NaN,NaN,2018-08-15 00:00:00,2018-08-15 16:39:01,1
4,940144190dcba6351888cafa43f3a3a5,00042b26cf59d7ce69dfabb4e55b4fd9,5,NaN,Gostei pois veio no prazo determinado .,2017-03-02 00:00:00,2017-03-03 10:54:59,1


### 2.3 Orders: Status breakdown & Missing dates

In [10]:
print(orders.info())
orders.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
None


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [11]:
query = """
SELECT order_status, count(*) as total_order,
    SUM(order_approved_at IS NULL) as not_approved,
    SUM(order_delivered_carrier_date IS NULL) as missing_carrier,
    SUM(order_delivered_customer_date IS NULL) as missing_customer
FROM orders
GROUP BY order_status
order by total_order desc

"""
order_status = pd.read_sql_query(query, conn)
order_status

,order_status,total_order,not_approved,missing_carrier,missing_customer
0,delivered,96478,14,2,8
1,shipped,1107,0,0,1107
2,canceled,625,141,550,619
3,unavailable,609,0,609,609
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,created,5,5,5,5
7,approved,2,0,2,2


In [12]:
orders_new= orders[orders['order_status']=='delivered']
orders_new = orders_new.dropna(subset=['order_approved_at','order_delivered_customer_date','order_delivered_carrier_date']).reset_index(drop=True)
orders_new.to_sql("orders_new", conn, if_exists="replace", index=False)

orders_new.info()


<class 'pandas.DataFrame'>
RangeIndex: 96455 entries, 0 to 96454
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       96455 non-null  str  
 1   customer_id                    96455 non-null  str  
 2   order_status                   96455 non-null  str  
 3   order_purchase_timestamp       96455 non-null  str  
 4   order_approved_at              96455 non-null  str  
 5   order_delivered_carrier_date   96455 non-null  str  
 6   order_delivered_customer_date  96455 non-null  str  
 7   order_estimated_delivery_date  96455 non-null  str  
dtypes: str(8)
memory usage: 5.9 MB


Keep only orders with status `deliveried` and all the dates are not empty

### 2.4 Orders: Timestamp consistency

In [13]:
query = """
SELECT
    COUNT(*) AS n_orders,

    SUM(order_approved_at            < order_purchase_timestamp)      AS approved_before_purchase,
    SUM(order_delivered_carrier_date < order_approved_at)             AS carrier_before_approved,
    SUM(order_delivered_customer_date < order_delivered_carrier_date) AS delivered_before_carrier,
    SUM(order_delivered_customer_date < order_purchase_timestamp)     AS delivered_before_purchase
FROM orders_new;
"""
order_anomalies = pd.read_sql_query(query, conn)
order_anomalies

,n_orders,approved_before_purchase,carrier_before_approved,delivered_before_carrier,delivered_before_purchase
0,96455,0,1350,23,0


There 1373 orders have timestamp not in order.

In [14]:
# Parse timestamp columns (you'll need real datetimes for delivery_days later anyway)
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders_new[date_cols] = orders_new[date_cols].apply(pd.to_datetime)

# Rows where the stages are out of order
bad_stages = (
    (orders_new["order_delivered_carrier_date"] < orders_new["order_approved_at"])
    | (orders_new["order_delivered_customer_date"] < orders_new["order_delivered_carrier_date"])
)

print(f"Orders before:  {len(orders_new):,}")
print(f"Dropped:        {bad_stages.sum():,} ({bad_stages.mean():.1%})")

orders_new = orders_new[~bad_stages].copy()
print(f"Orders after:   {len(orders_new):,}")

Orders before:  96,455
Dropped:        1,373 (1.4%)
Orders after:   95,082


In [15]:
orders_new.to_sql("orders_new_clean", conn, if_exists="replace", index=False)

95082

**Findings**
- 96,478 orders (97.0%) are `delivered`; 23 of them are missing at least one
  timestamp (approval, carrier handoff, or customer delivery).
- 6 `canceled` orders have a delivery date (likely returns or data errors).
- 1,107 orders (1.1%) are `shipped` but never received a delivery date.
- Among delivered orders with all timestamps, none was delivered before purchase,
  but `1,373` (≈1.4%) have stages out of order (mostly handed to the carrier
  before payment approval).

**Decision:** 
- Keep `delivered` orders with all three timestamps present and stages
in chronological order: `95082` orders (`97`% of all 99,441 orders),
saved as `new_orders`. This keeps every stage duration (seller handling,
carrier transit) non-negative.

**Limitations**
- Excluded `shipped` orders are likely the slowest deliveries (right-censoring),
  so delivery times and late rates may be slightly understated.
- Orders dropped for out-of-order stages still had valid end-to-end delivery
  times; dropping them is a conservative choice for the stage analysis.

### 2.5 Order items: items and sellers per

`order_items` has one row per item. Before aggregating to one row per order,
check how many orders have multiple items and how many have multiple sellers.

In [16]:
query = """
SELECT n_items, count(*) AS n_orders,  avg_freight
FROM (
    SELECT order_id, MAX(order_item_id) AS n_items, round(avg(freight_value), 2) AS avg_freight
    FROM order_items
    GROUP BY order_id    
)
GROUP BY n_items
ORDER BY n_items;



"""
result = pd.read_sql_query(query, conn)
result

,n_items,n_orders,avg_freight
0,1,88863,13.29
1,2,7516,13.37
2,3,1322,15.10
3,4,505,8.40
4,5,204,11.85
5,6,198,30.44
6,7,22,9.86
7,8,8,28.01
8,9,3,50.97
9,10,8,9.98


In [17]:
query = """
WITH per_order AS (
    SELECT order_id,
           COUNT(*)               AS n_items,
           COUNT(DISTINCT seller_id) AS n_sellers
    FROM order_items
    GROUP BY order_id
)
SELECT n_sellers,
       COUNT(*) AS n_orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM per_order
GROUP BY n_sellers
ORDER BY n_sellers;
"""
result = pd.read_sql_query(query, conn)
result

,n_sellers,n_orders,pct
0,1,97388,98.70
1,2,1219,1.24
2,3,54,0.05
3,4,3,0.00
4,5,2,0.00


**Findings**
- 98,666 orders have items (775 orders in `orders` have none).
- 9,803 orders (9.9%) contain more than one item, so item-level columns
  (freight, weight) must be aggregated to one row per order.
- 1,278 orders (1.3%) have more than one seller, so `seller_state` is ambiguous
  for them.

**Decision:** keep all rows in `order_items`. In the join, items are aggregated
to one row per order (item count, total freight, total weight).

For the 1.3% of orders with more than one seller, `seller_state` is set to NULL; these orders
are kept for delivery-time analysis and excluded only from seller-level analysis.

##  3: Build the order-level analysis table

### 3.1 Items: one row per order 

`order_items` has one row per item. Aggregate it, together with `products` and
`sellers`, into one row per order, saved as `items_by_order`.

| Column           | Description                                   | Source               | Aggregation                         |
|------------------|-----------------------------------------------|----------------------|-------------------------------------|
| `order_id`       | Order key (unique)                            | `order_items`        | `GROUP BY`                          |
| `n_items`        | Number of items in the order                  | `order_items`        | `COUNT(*)`                          |
| `n_sellers`      | Number of distinct sellers                    | `order_items`        | `COUNT(DISTINCT seller_id)`         |
| `total_price`    | Total item price (BRL)                        | `order_items`        | `SUM(price)`                        |
| `total_freight`  | Total freight cost (BRL)                      | `order_items`        | `SUM(freight_value)`                |
| `total_weight_g` | Total product weight (grams)                  | `products`           | `SUM(product_weight_g)`             |
| `seller_state`   | Seller's state; NULL if more than one seller  | `sellers`            | value if `n_sellers = 1`, else NULL |

**Expected:** 98,666 rows (one per order in `order_items`), `order_id` unique.

In [18]:
query = """
SELECT 
    oi.order_id,
    COUNT(*) as n_items,
    COUNT(DISTINCT oi.seller_id) as n_sellers,
    ROUND(SUM(oi.price), 2) as total_price,
    ROUND(SUM(oi.freight_value), 2) as total_freight,
    CASE
        WHEN COUNT(p.product_weight_g) < COUNT(*) THEN NULL
        ELSE SUM(p.product_weight_g)
    END AS total_weight_g,
    CASE
        WHEN COUNT(DISTINCT oi.seller_id) > 1 THEN NULL
        ELSE s.seller_state
    END as seller_state
FROM order_items oi
LEFT JOIN products p ON oi.product_id = p.product_id
LEFT JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY oi.order_id
    
"""
result = pd.read_sql_query(query, conn)
result.to_sql('items_by_order', conn, if_exists='replace', index=False)

result.head(10)

,order_id,n_items,n_sellers,total_price,total_freight,total_weight_g,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,58.90,13.29,650.0,SP
1,00018f77f2f0320c557190d7a144bdd3,1,1,239.90,19.93,30000.0,SP
2,000229ec398224ef6ca0657da4fc703e,1,1,199.00,17.87,3050.0,MG
3,00024acbcdf0a6daa1e931b038114c75,1,1,12.99,12.79,200.0,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,199.90,18.14,3750.0,PR
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,1,21.90,12.69,450.0,SP
6,00054e8431b9d7675808bcb819fb4a32,1,1,19.90,11.85,200.0,SP
7,000576fe39319847cbb9d288c5617fa6,1,1,810.00,70.75,13805.0,SP
8,0005a1a1728c9d785b8e2b08b904576c,1,1,145.95,11.65,2000.0,SP
9,0005f50442cb953dcd1d21e1fb923495,1,1,53.99,11.40,850.0,SP


In [19]:
query = """
SELECT
    COUNT(*)                      AS n_rows,             
    COUNT(DISTINCT order_id)      AS n_orders,           
    SUM(n_items)                  AS total_items,       
    SUM(seller_state IS NULL)     AS null_seller_state,  
    SUM(total_weight_g IS NULL)   AS null_weight
FROM items_by_order
    
"""
result = pd.read_sql_query(query, conn)
result

,n_rows,n_orders,total_items,null_seller_state,null_weight
0,98666,98666,112650,1278,16


**Result:** `items_by_order` has 98,666 rows, one per order (`order_id` unique);
item counts sum to 112,650, so no items were lost in the joins.
- `1,278` orders have more than one seller → `seller_state` is NULL.
- `16` orders have at least one item without a weight → `total_weight_g`
  is NULL rather than understated. Only the regression (Q5) uses weight.

### 3.2 Reviews: one row per order  

`order_reviews` has 547 orders with more than one review (see 2.1–2.2).
Keep only the **latest** review per order (by `review_answer_timestamp`),
saved as `reviews_by_order`.

| Column                    | Description                                      | Source          | Rule                                              |
|---------------------------|--------------------------------------------------|-----------------|---------------------------------------------------|
| `order_id`                | Order key (unique)                               | `order_reviews` | one row per order                                 |
| `review_score`            | Customer rating, 1–5, from the latest review     | `order_reviews` | `ROW_NUMBER() ... ORDER BY review_answer_timestamp DESC` = 1 |
| `review_answer_timestamp` | When the customer answered the latest review     | `order_reviews` | same row as `review_score`                        |
| `n_reviews`               | Number of reviews the order received             | `order_reviews` | `COUNT(*) OVER (PARTITION BY order_id)`           |

**Expected:** 98,673 rows (one per reviewed order), `order_id` unique.
Orders without a review are not in this table; they get NULL after the
`LEFT JOIN` in 3.3.

In [20]:
query = """
SELECT order_id, review_score, review_answer_timestamp, n_reviews
FROM (
    SELECT
        order_id,
        review_score,
        review_answer_timestamp,
        COUNT(*) OVER (PARTITION BY order_id) AS n_reviews,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY review_answer_timestamp DESC, review_id
        ) AS rn
    FROM order_reviews
)
WHERE rn = 1;
"""
result = pd.read_sql_query(query, conn)
result.to_sql("reviews_by_order", conn, if_exists="replace", index=False)

98673

In [21]:
pd.read_sql("""
SELECT
    COUNT(*)                  AS n_rows,          -- expect 98,673
    COUNT(DISTINCT order_id)  AS n_orders,        -- expect = n_rows
    SUM(n_reviews > 1)        AS multi_review,    -- expect 547
    SUM(review_score IS NULL) AS null_score       -- expect 0
FROM reviews_by_order
""", conn)

,n_rows,n_orders,multi_review,null_score
0,98673,98673,547,0


**Result:** `reviews_by_order` has 98,673 rows, one per reviewed order
(`order_id` unique). For the 547 orders with several reviews, only the latest
was kept (`n_reviews` records how many there were). No missing scores.

### 3.3 Join all tables 

Join the cleaned orders with customers, items and reviews into one analysis
table, saved as `analysis`. `orders_clean` is the base table, so the filters
from Section 2 define which orders are included.

| Column                          | Description                                  | Source             | Join        |
|---------------------------------|----------------------------------------------|--------------------|-------------|
| `order_id`                      | Order key (unique)                           | `orders_new_clean`     | base table  |
| `customer_id`                   | Per-order customer key                       | `orders_new_clean`     | base table  |
| `order_purchase_timestamp`      | When the order was placed                    | `orders_new_clean`     | base table  |
| `order_approved_at`             | When payment was approved                    | `orders_new_clean`     | base table  |
| `order_delivered_carrier_date`  | When the seller handed it to the carrier     | `orders_new_clean`     | base table  |
| `order_delivered_customer_date` | When the customer received it                | `orders_new_clean`     | base table  |
| `order_estimated_delivery_date` | Delivery date promised to the customer       | `orders_new_clean`     | base table  |
| `customer_unique_id`            | Real customer (same across repeat orders)    | `customers`        | `JOIN`      |
| `customer_state`                | Customer's state                             | `customers`        | `JOIN`      |
| `n_items`                       | Number of items                              | `items_by_order`   | `LEFT JOIN` |
| `n_sellers`                     | Number of distinct sellers                   | `items_by_order`   | `LEFT JOIN` |
| `total_price`                   | Total item price (BRL)                       | `items_by_order`   | `LEFT JOIN` |
| `total_freight`                 | Total freight cost (BRL)                     | `items_by_order`   | `LEFT JOIN` |
| `total_weight_g`                | Total weight (g); NULL if any item is missing | `items_by_order`  | `LEFT JOIN` |
| `seller_state`                  | Seller's state; NULL if more than one seller | `items_by_order`   | `LEFT JOIN` |
| `review_score`                  | Latest review score (1–5); NULL if no review | `reviews_by_order` | `LEFT JOIN` |
| `n_reviews`                     | Number of reviews; NULL if no review         | `reviews_by_order` | `LEFT JOIN` |


In [ ]:
query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    c.customer_unique_id,
    c.customer_state,
    i.n_items,
    i.n_sellers,
    i.total_price,
    i.total_freight,
    i.total_weight_g,
    i.seller_state,
    r.review_score,
    r.review_answer_timestamp,
    r.n_reviews
FROM orders_new_clean o
LEFT JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN items_by_order i ON o.order_id = i.order_id
LEFT JOIN reviews_by_order r ON o.order_id = r.order_id;
"""
analysis = pd.read_sql_query(query, conn)
analysis.head(10)

,order_id,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,n_items,n_sellers,total_price,total_freight,total_weight_g,seller_state,review_score,review_answer_timestamp,n_reviews
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,SP,1,1,29.99,8.72,500.0,SP,4.0,2017-10-12 03:43:48,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,BA,1,1,118.70,22.76,400.0,SP,4.0,2018-08-08 18:37:50,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,GO,1,1,159.90,19.22,420.0,SP,5.0,2018-08-22 19:07:58,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,RN,1,1,45.00,27.20,450.0,MG,5.0,2017-12-05 19:21:58,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,SP,1,1,19.90,8.72,250.0,SP,5.0,2018-02-18 13:02:51,1.0
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,80bb27c7c16e8f973207a5086ab329e2,PR,1,1,147.90,27.36,7150.0,SP,4.0,2017-07-27 22:48:30,1.0
6,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,932afa1e708222e5821dac9cd5db4cae,RJ,1,1,59.99,15.17,50.0,SP,5.0,2017-05-28 02:59:57,1.0
7,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,39382392765b6dc74812866ee5ee92a7,RS,1,1,19.90,16.05,300.0,SP,1.0,2017-02-05 01:58:35,1.0
8,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,299905e3934e9e181bfb2e164dd4b4f8,SP,1,1,149.99,19.77,9750.0,SP,5.0,2017-08-18 01:47:32,1.0
9,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07 00:00:00,f2a85dec752b8517b5e58a06ff3cd937,RJ,2,1,198.00,61.06,18000.0,ES,1.0,2017-05-30 23:13:47,1.0


In [26]:
base_rows = pd.read_sql("SELECT COUNT(*) AS n FROM orders_new_clean", conn)["n"].iloc[0]

checks = pd.read_sql("""
SELECT
    COUNT(*)                     AS n_rows,
    COUNT(DISTINCT order_id)     AS n_orders,
    SUM(customer_state IS NULL)  AS null_customer_state,
    SUM(n_items IS NULL)         AS no_items,
    SUM(review_score IS NULL)    AS no_review,
    SUM(seller_state IS NULL)    AS null_seller_state,
    SUM(total_weight_g IS NULL)  AS null_weight
FROM analysis
""", conn)

print(f"orders_clean rows: {base_rows:,}")
checks

orders_clean rows: 95,082


,n_rows,n_orders,null_customer_state,no_items,no_review,null_seller_state,null_weight
0,95082,95082,0,0,639,1223,16


**Result:** `analysis` has 95,082 rows, exactly one per order in `orders_new_clean`,
with no missing customer data.
- Every delivered order has at least one item (the 775 orders without items were
  all non-delivered and removed in Section 2).
- 639 orders (0.7%) have no review → `review_score` is NULL; excluded from H1 only.
- 1,223 orders (1.3%) have several sellers → `seller_state` is NULL; excluded from
  seller-level analysis only.
- 16 orders have no product weight → excluded from weight-based regression only.

## 4. Feature engineering


In [29]:
analysis.columns

Index(['order_id', 'customer_id', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_state', 'n_items', 'n_sellers',
       'total_price', 'total_freight', 'total_weight_g', 'seller_state',
       'review_score', 'review_answer_timestamp', 'n_reviews'],
      dtype='str')


Add derived columns to `analysis`. All durations are in days.

| Feature                | Definition                                              | Used in |
|------------------------|---------------------------------------------------------|---------|
| `delivery_days`        | delivered − purchased                                   | Q1, Q2, Q5 |
| `delay_days`           | delivered date − estimated date (negative = early)      | Q2, Q4 |
| `is_late`              | 1 if delivered **after** the estimated date, else 0     | Q3, Q5 |
| `seller_days`          | handed to carrier − approved (seller handling)          | stage analysis |
| `carrier_days`         | delivered − handed to carrier (transit)                 | stage analysis |
| `same_state`           | 1 if `customer_state = seller_state`, NULL if multi-seller | Q5 |
| `purchase_month`       | `YYYY-MM` of purchase                                   | EDA trend, Q5 |
| `purchase_weekday`     | day of week of purchase                                 | EDA |

In [35]:
query = """
WITH d AS (
    SELECT
        *,
        julianday(date(order_purchase_timestamp))       AS purchase_d,
        julianday(date(order_approved_at))              AS approved_d,
        julianday(date(order_delivered_carrier_date))   AS carrier_d,
        julianday(date(order_delivered_customer_date))  AS delivered_d,
        julianday(date(order_estimated_delivery_date))  AS estimated_d
    FROM analysis
)
SELECT
    order_id, customer_id, customer_unique_id, customer_state,
    order_purchase_timestamp, order_approved_at, order_delivered_carrier_date,
    order_delivered_customer_date, order_estimated_delivery_date,
    n_items, n_sellers, total_price, total_freight, total_weight_g, seller_state,
    review_score, review_answer_timestamp, n_reviews,

    CAST(delivered_d - purchase_d  AS INTEGER) AS delivery_days,
    CAST(delivered_d - estimated_d AS INTEGER) AS delay_days,
    (delivered_d > estimated_d)                AS is_late,
    CAST(carrier_d   - approved_d  AS INTEGER) AS seller_days,
    CAST(delivered_d - carrier_d   AS INTEGER) AS carrier_days,
    (customer_state = seller_state)            AS same_state,

    strftime('%Y-%m', order_purchase_timestamp) AS purchase_month,
    CASE CAST(strftime('%w', order_purchase_timestamp) AS INTEGER)
        WHEN 0 THEN 'Sunday'    WHEN 1 THEN 'Monday'   WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday' WHEN 4 THEN 'Thursday' WHEN 5 THEN 'Friday'
        ELSE 'Saturday'
    END AS purchase_weekday
FROM d;
"""
analysis = pd.read_sql_query(query, conn)
analysis.head()

,order_id,customer_id,customer_unique_id,customer_state,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,n_items,...,review_answer_timestamp,n_reviews,delivery_days,delay_days,is_late,seller_days,carrier_days,same_state,purchase_month,purchase_weekday
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,SP,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,...,2017-10-12 03:43:48,1.0,8,-8,0,2,6,1.0,2017-10,Monday
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,BA,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,...,2018-08-08 18:37:50,1.0,14,-6,0,0,12,0.0,2018-07,Tuesday
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,GO,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,...,2018-08-22 19:07:58,1.0,9,-18,0,0,9,0.0,2018-08,Wednesday
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,RN,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,...,2017-12-05 19:21:58,1.0,14,-13,0,4,10,0.0,2017-11,Saturday
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,SP,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,...,2018-02-18 13:02:51,1.0,3,-10,0,1,2,1.0,2018-02,Tuesday


In [43]:
print(analysis[["delivery_days", "delay_days", "seller_days", "carrier_days"]].describe())
print("\nLate rate:", round(analysis["is_late"].mean(), 4))
print("\n", analysis["same_state"].value_counts(dropna=False))
print(f'\nTotal Orders with delivery_days > 60: {(analysis["delivery_days"] > 60).sum()}')
print(f'Total Orders with delay_days < -60: {(analysis["delay_days"] < -60).sum()}')

       delivery_days    delay_days   seller_days  carrier_days
count   95082.000000  95082.000000  95082.000000  95082.000000
mean       12.557035    -11.824236      2.750962      9.317011
std         9.580259     10.198932      3.505158      8.786438
min         0.000000   -147.000000      0.000000      0.000000
25%         7.000000    -17.000000      1.000000      4.000000
50%        10.000000    -12.000000      2.000000      7.000000
75%        16.000000     -7.000000      4.000000     12.000000
max       210.000000    188.000000    126.000000    205.000000

Late rate: 0.0685

 same_state
0.0    60118
1.0    33741
NaN     1223
Name: count, dtype: int64

Total Orders with delivery_days > 60: 296
Total Orders with delay_days < -60: 33


**Result** (95,082 orders; durations in calendar days):
- Median delivery time is 10 days (mean 12.6, max 210): right-skewed with a long tail.
- The typical order arrives 12 days **before** the estimated date, yet 6.85% are late.
- Carrier transit (mean 9.3 days) takes about three times longer than seller
  handling (mean 2.8 days).
- 64% of orders with a single seller ship across state lines (`same_state = 0`).
- A few extreme values (up to 210 days) are kept for now and reviewed in EDA.
- 296 orders (0.3%) took more than 60 days and 33 (0.03%) arrived over 60 days
  early. 
- They are kept: the long tail is real and is the subject of Q1. Robust
  summaries (median, bootstrap CIs of the median) are used where outliers matter.

## 5. Summary & save
